# Example 9.5.4 / 1: Read from a View and Write Results

Companion notebook for **9.5.4 SQLite** in the AMPL lab Documentation, section "Read from a View and Write Results". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read from a View and Write Results

`planning.db` holds two related tables: `products` (rate and profit) and `limits` (market limit
and an `active` flag). A view `model_input` joins them and keeps only active products, so the
AMPL side sees one clean input table:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # maximum sales, tons
param avail >= 0 default 40;      # hours available

var Make {p in PROD} >= 0, <= market[p];
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

table Input IN "amplsqlite3" "planning.db" "model_input":
    PROD <- [product], rate, profit, market ~ max_sales;
table Plan OUT "amplsqlite3" "planning.db" "plan" "write=drop":
    {p in PROD} -> [product], Make[p] ~ make;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import os, sqlite3
if os.path.exists("planning.db"):
    os.remove("planning.db")
db = sqlite3.connect("planning.db")
db.executescript("""
CREATE TABLE products (product TEXT PRIMARY KEY, rate REAL, profit REAL);
CREATE TABLE limits (product TEXT PRIMARY KEY, max_sales REAL, active INTEGER);
INSERT INTO products VALUES ('bands', 200, 25), ('coils', 140, 30),
                            ('plate', 160, 29), ('wire', 120, 40);
INSERT INTO limits VALUES ('bands', 6000, 1), ('coils', 4000, 1),
                          ('plate', 3500, 1), ('wire', 900, 0);
CREATE VIEW model_input AS
    SELECT p.product, p.rate, p.profit, l.max_sales
    FROM products p JOIN limits l ON p.product = l.product
    WHERE l.active = 1;
""")
db.commit()
db.close()

In [5]:
# run.py
ampl.eval("load amplsqlite3.dll;")
ampl.read_table("Input")
print(ampl.set["PROD"].to_list())
ampl.solve(solver="highs", verbose=False)
if ampl.solve_result == "solved":
    ampl.write_table("Plan")
db = sqlite3.connect("planning.db")
print(db.execute("SELECT * FROM plan").fetchall())
db.close()

['bands', 'coils', 'plate']
[('bands', 6000.0), ('coils', 0.0), ('plate', 1600.0)]


### Pure AMPL: model.run

The inactive product `wire` never reaches AMPL, because the view filters it out; `~ max_sales`
keeps the model's name `market` while the view keeps its own column name. `write=drop`
recreates the `plan` table on every run. With the default `write=delete` the handler instead
keeps an existing table's definition and deletes its rows before writing.

Pure AMPL, with the same `model.mod` and `planning.db`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
import os, sqlite3
if os.path.exists("planning.db"):
    os.remove("planning.db")
db = sqlite3.connect("planning.db")
db.executescript("""
CREATE TABLE products (product TEXT PRIMARY KEY, rate REAL, profit REAL);
CREATE TABLE limits (product TEXT PRIMARY KEY, max_sales REAL, active INTEGER);
INSERT INTO products VALUES ('bands', 200, 25), ('coils', 140, 30),
                            ('plate', 160, 29), ('wire', 120, 40);
INSERT INTO limits VALUES ('bands', 6000, 1), ('coils', 4000, 1),
                          ('plate', 3500, 1), ('wire', 900, 0);
CREATE VIEW model_input AS
    SELECT p.product, p.rate, p.profit, l.max_sales
    FROM products p JOIN limits l ON p.product = l.product
    WHERE l.active = 1;
""")
db.commit()
db.close()

In [8]:
%%ampl_eval
# model.run
load amplsqlite3.dll;
read table Input;
display PROD;
option solver highs;
option solver_msg 0;
solve;
if solve_result = "solved" then {
    write table Plan;
}
display Make;

set PROD := bands coils plate;

Make [*] :=
bands  6000
coils     0
plate  1600
;

